In [ ]:
import pandas as pd

df_all_shuffled_training_mass = pd.read_csv("track_based_table.csv") # training sample

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
#------------------------------------------
# 1. Load and Prepare the Data
#------------------------------------------

X=df_all_shuffled_training_mass[['El', 'Mu', 'Pi', 'Ka', 'Pr', 'N Tracks']].values
# Extract mass separately
mass = df['mass'].values

mass=df_all_shuffled_training_mass['InvariantMass'].values
# Create a train/test split

X_train, X_test, mass_train, mass_test = train_test_split(X, mass, test_size=0.05, random_state=42)

from sklearn.preprocessing import MinMaxScaler
#Initialize scaler

scaler = MinMaxScaler()
# Fit and transform training data, transform test data

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(X_train.shape, X_test.shape)

import tensorflow as tf
print("TensorFlow version:", tf.version)
print("Available devices:", tf.config.list_physical_devices())
tf.debugging.set_log_device_placement(True)

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1" # Disable GPU (if any)
#------------------------------------------
# 2. Define the Autoencoder Model
#------------------------------------------

input_dim = X_train.shape[1] # should be 6
encoding_dim = 4 # latent space dimension
# Input layer

input_layer = Input(shape=(input_dim,))
# Encoding layer (compression)

encoded = Dense(encoding_dim, activation='relu')(input_layer)
# Decoding layer (reconstruction)

decoded = Dense(input_dim, activation='sigmoid')(encoded)
# Autoencoder model

autoencoder = Model(inputs=input_layer, outputs=decoded)
autoencoder.compile(optimizer='adam', loss='mse')

from tensorflow.keras.optimizers import Adam
autoencoder.compile(optimizer=Adam(learning_rate=0.01), loss='mse')
autoencoder.compile(optimizer=Adam(learning_rate=0.001), loss='mse') # best, 185 epochs automatic stop
#------------------------------------------
# 3. Train the Autoencoder
#------------------------------------------

early_stopping = EarlyStopping(
monitor='val_loss',
patience=2,
restore_best_weights=True
)

history = autoencoder.fit(X_train, X_train,
epochs=500,
# epochs=500,
batch_size=256,
# batch_size=64,
shuffle=True,
validation_data=(X_test, X_test),
verbose=2,
callbacks=[early_stopping]
)
with tf.device('/CPU:0'):
history = autoencoder.fit(X_train, X_train,
epochs=500,
batch_size=64,
validation_data=(X_test, X_test),
callbacks=[early_stopping],
verbose=2)
print(f"Training stopped at epoch {len(history.history['loss'])}")

print(f"Training stopped at epoch {len(history.history['loss'])}")
